# Insighta NLP Training (DistilBERT, Colab GPU)

This notebook trains a **multi-task DistilBERT** model for:
- sentiment (3 classes)
- detectedIntent (8 classes)
- issueType (10 classes)
- priority (3 classes)

It includes:
- class-weighted loss for sentiment/priority
- weighted multi-task objective
- run-versioned output folder
- post-training temperature calibration
- hard-case subset metrics


In [ ]:
# Colab setup
!pip -q install -U transformers datasets evaluate scikit-learn accelerate


In [ ]:
import json
import random
import shutil
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, List

import numpy as np
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss

from datasets import Dataset
from google.colab import drive
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoConfig,
    AutoTokenizer,
    DistilBertModel,
    DistilBertPreTrainedModel,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Reproducibility
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# Mount Drive and configure paths
drive.mount('/content/drive')

# Update BASE_DIR to where your repo/data lives in Drive
BASE_DIR = Path('/content/drive/MyDrive/Insighta')
DATA_DIR = BASE_DIR / 'data' / 'nlp'

RUN_ID = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = BASE_DIR / 'artifacts' / f'distilbert_multitask_{RUN_ID}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train.jsonl'
VAL_PATH = DATA_DIR / 'val.jsonl'
TEST_PATH = DATA_DIR / 'test.jsonl'

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f'Missing dataset file: {p}')

print('RUN_ID:', RUN_ID)
print('Data dir:', DATA_DIR)
print('Output dir:', OUTPUT_DIR)


In [ ]:
# Load JSONL rows
def load_jsonl(path: Path) -> List[Dict]:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(TRAIN_PATH)
val_rows = load_jsonl(VAL_PATH)
test_rows = load_jsonl(TEST_PATH)

print('train:', len(train_rows), 'val:', len(val_rows), 'test:', len(test_rows))
print('sample metadata:', test_rows[0].get('metadata', {}))


In [ ]:
# Label vocab (taxonomy-locked)
SENTIMENT_LABELS = ['Negative', 'Neutral', 'Positive']
INTENT_LABELS = [
    'General Complaint',
    'Request Status Update',
    'Request Refund',
    'Request Cancellation',
    'Appeal Claim Decision',
    'Report Billing Error',
    'Report Policy Change Issue',
    'Report Document Processing Delay',
    'Share Positive Feedback',
]
ISSUE_LABELS = [
    'Claim Denial',
    'Billing Dispute',
    'Policy Cancellation',
    'Policy Update Issue',
    'Payment Issue',
    'Document Processing Delay',
    'Technical Issue',
    'Fraud Report',
    'Delivery Issue',
    'Uncategorized',
    'Positive Feedback',
]
PRIORITY_LABELS = ['Low', 'Medium', 'High']

sentiment2id = {v: i for i, v in enumerate(SENTIMENT_LABELS)}
intent2id = {v: i for i, v in enumerate(INTENT_LABELS)}
issue2id = {v: i for i, v in enumerate(ISSUE_LABELS)}
priority2id = {v: i for i, v in enumerate(PRIORITY_LABELS)}


def encode_rows(rows: List[Dict]) -> List[Dict]:
    encoded = []
    for row in rows:
        labels = row['labels']
        s = labels['sentiment']
        i = labels['detectedIntent']
        it = labels['issueType']
        p = labels['priority']

        if s not in sentiment2id:
            raise ValueError(f'Unknown sentiment: {s}')
        if i not in intent2id:
            raise ValueError(f'Unknown intent: {i}')
        if it not in issue2id:
            raise ValueError(f'Unknown issueType: {it}')
        if p not in priority2id:
            raise ValueError(f'Unknown priority: {p}')

        encoded.append({
            'text': row['text'],
            'labels_sentiment': sentiment2id[s],
            'labels_intent': intent2id[i],
            'labels_issue': issue2id[it],
            'labels_priority': priority2id[p],
        })
    return encoded

train_enc = encode_rows(train_rows)
val_enc = encode_rows(val_rows)
test_enc = encode_rows(test_rows)

# Keep metadata for hard-case analysis
val_meta = [r.get('metadata', {}) for r in val_rows]
test_meta = [r.get('metadata', {}) for r in test_rows]

print('Encoded rows ready.')
print('Sample:', train_enc[0])


In [ ]:
# Save label maps for backend integration
label_maps = {
    'sentiment': {'id2label': {str(i): v for i, v in enumerate(SENTIMENT_LABELS)}, 'label2id': sentiment2id},
    'intent': {'id2label': {str(i): v for i, v in enumerate(INTENT_LABELS)}, 'label2id': intent2id},
    'issueType': {'id2label': {str(i): v for i, v in enumerate(ISSUE_LABELS)}, 'label2id': issue2id},
    'priority': {'id2label': {str(i): v for i, v in enumerate(PRIORITY_LABELS)}, 'label2id': priority2id},
}
for name, mapping in label_maps.items():
    (OUTPUT_DIR / f'label_maps_{name}.json').write_text(json.dumps(mapping, indent=2), encoding='utf-8')
print('Wrote label maps to', OUTPUT_DIR)


In [ ]:
# Build datasets + tokenizer
train_ds = Dataset.from_list(train_enc)
val_ds = Dataset.from_list(val_enc)
test_ds = Dataset.from_list(test_enc)

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 320

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

print(train_ds)


In [ ]:
# Class weights for sentiment/priority

def compute_inverse_freq_weights(labels, num_classes: int):
    counts = np.bincount(labels, minlength=num_classes).astype(np.float32)
    counts = np.maximum(counts, 1.0)
    inv = 1.0 / counts
    weights = inv / inv.mean()
    return weights

sentiment_train = np.array([r['labels_sentiment'] for r in train_enc], dtype=np.int64)
priority_train = np.array([r['labels_priority'] for r in train_enc], dtype=np.int64)

sentiment_class_weights = compute_inverse_freq_weights(sentiment_train, len(SENTIMENT_LABELS))
priority_class_weights = compute_inverse_freq_weights(priority_train, len(PRIORITY_LABELS))

LOSS_WEIGHTS = {
    'sentiment': 0.35,
    'intent': 0.25,
    'issue': 0.15,
    'priority': 0.25,
}

print('sentiment class weights:', sentiment_class_weights)
print('priority class weights:', priority_class_weights)
print('loss weights:', LOSS_WEIGHTS)


In [ ]:
# Data collator
@dataclass
class MultiTaskDataCollator:
    tokenizer: object

    def __call__(self, features):
        label_keys = ['labels_sentiment', 'labels_intent', 'labels_issue', 'labels_priority']
        labels = {k: [f[k] for f in features] for k in label_keys}
        clean_features = []
        for f in features:
            g = dict(f)
            for k in label_keys:
                g.pop(k, None)
            # keep only model fields
            g = {k: v for k, v in g.items() if k in {'input_ids', 'attention_mask', 'token_type_ids'}}
            clean_features.append(g)

        batch = self.tokenizer.pad(clean_features, return_tensors='pt')
        batch['labels_sentiment'] = torch.tensor(labels['labels_sentiment'], dtype=torch.long)
        batch['labels_intent'] = torch.tensor(labels['labels_intent'], dtype=torch.long)
        batch['labels_issue'] = torch.tensor(labels['labels_issue'], dtype=torch.long)
        batch['labels_priority'] = torch.tensor(labels['labels_priority'], dtype=torch.long)
        return batch

collator = MultiTaskDataCollator(tokenizer=tokenizer)


In [ ]:
# Multi-task DistilBERT model
class MultiTaskDistilBert(DistilBertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels_sentiment = config.num_labels_sentiment
        self.num_labels_intent = config.num_labels_intent
        self.num_labels_issue = config.num_labels_issue
        self.num_labels_priority = config.num_labels_priority

        self.distilbert = DistilBertModel(config)
        self.dropout = nn.Dropout(config.seq_classif_dropout)

        hidden = config.dim
        self.classifier_sentiment = nn.Linear(hidden, self.num_labels_sentiment)
        self.classifier_intent = nn.Linear(hidden, self.num_labels_intent)
        self.classifier_issue = nn.Linear(hidden, self.num_labels_issue)
        self.classifier_priority = nn.Linear(hidden, self.num_labels_priority)

        self.loss_weight_sentiment = getattr(config, 'loss_weight_sentiment', 0.35)
        self.loss_weight_intent = getattr(config, 'loss_weight_intent', 0.25)
        self.loss_weight_issue = getattr(config, 'loss_weight_issue', 0.15)
        self.loss_weight_priority = getattr(config, 'loss_weight_priority', 0.25)

        sentiment_weights = torch.tensor(getattr(config, 'sentiment_class_weights', [1.0] * self.num_labels_sentiment), dtype=torch.float)
        priority_weights = torch.tensor(getattr(config, 'priority_class_weights', [1.0] * self.num_labels_priority), dtype=torch.float)

        self.register_buffer('sentiment_class_weights', sentiment_weights)
        self.register_buffer('priority_class_weights', priority_weights)

        self.post_init()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels_sentiment=None,
        labels_intent=None,
        labels_issue=None,
        labels_priority=None,
        **kwargs,
    ):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs.last_hidden_state
        pooled = hidden_state[:, 0]
        pooled = self.dropout(pooled)

        logits_sentiment = self.classifier_sentiment(pooled)
        logits_intent = self.classifier_intent(pooled)
        logits_issue = self.classifier_issue(pooled)
        logits_priority = self.classifier_priority(pooled)

        loss = None
        if all(x is not None for x in [labels_sentiment, labels_intent, labels_issue, labels_priority]):
            ce_sent = CrossEntropyLoss(weight=self.sentiment_class_weights)
            ce_intent = CrossEntropyLoss()
            ce_issue = CrossEntropyLoss()
            ce_priority = CrossEntropyLoss(weight=self.priority_class_weights)

            l_sent = ce_sent(logits_sentiment, labels_sentiment)
            l_intent = ce_intent(logits_intent, labels_intent)
            l_issue = ce_issue(logits_issue, labels_issue)
            l_priority = ce_priority(logits_priority, labels_priority)

            loss = (
                self.loss_weight_sentiment * l_sent
                + self.loss_weight_intent * l_intent
                + self.loss_weight_issue * l_issue
                + self.loss_weight_priority * l_priority
            )

        return {
            'loss': loss,
            'logits': (logits_sentiment, logits_intent, logits_issue, logits_priority),
        }

config = AutoConfig.from_pretrained(MODEL_NAME)
config.num_labels_sentiment = len(SENTIMENT_LABELS)
config.num_labels_intent = len(INTENT_LABELS)
config.num_labels_issue = len(ISSUE_LABELS)
config.num_labels_priority = len(PRIORITY_LABELS)

config.loss_weight_sentiment = LOSS_WEIGHTS['sentiment']
config.loss_weight_intent = LOSS_WEIGHTS['intent']
config.loss_weight_issue = LOSS_WEIGHTS['issue']
config.loss_weight_priority = LOSS_WEIGHTS['priority']

config.sentiment_class_weights = sentiment_class_weights.tolist()
config.priority_class_weights = priority_class_weights.tolist()

model = MultiTaskDistilBert.from_pretrained(MODEL_NAME, config=config)
print('Model initialized')


In [ ]:
# Metrics for Trainer

def compute_metrics(eval_pred):
    preds = eval_pred.predictions
    labels = eval_pred.label_ids

    if not isinstance(preds, (tuple, list)) or len(preds) != 4:
        raise ValueError('Expected 4 prediction tensors for multi-task outputs.')
    if not isinstance(labels, (tuple, list)) or len(labels) != 4:
        raise ValueError('Expected 4 label tensors for multi-task outputs.')

    y_sent, y_intent, y_issue, y_prio = labels
    p_sent = np.argmax(preds[0], axis=-1)
    p_intent = np.argmax(preds[1], axis=-1)
    p_issue = np.argmax(preds[2], axis=-1)
    p_prio = np.argmax(preds[3], axis=-1)

    metrics = {
        'sentiment_acc': accuracy_score(y_sent, p_sent),
        'intent_acc': accuracy_score(y_intent, p_intent),
        'issue_acc': accuracy_score(y_issue, p_issue),
        'priority_acc': accuracy_score(y_prio, p_prio),
        'sentiment_macro_f1': f1_score(y_sent, p_sent, average='macro'),
        'intent_macro_f1': f1_score(y_intent, p_intent, average='macro'),
        'issue_macro_f1': f1_score(y_issue, p_issue, average='macro'),
        'priority_macro_f1': f1_score(y_prio, p_prio, average='macro'),
    }
    metrics['macro_f1_avg'] = float(np.mean([
        metrics['sentiment_macro_f1'],
        metrics['intent_macro_f1'],
        metrics['issue_macro_f1'],
        metrics['priority_macro_f1'],
    ]))
    return metrics


In [ ]:
# Training config
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,
    weight_decay=0.01,
    logging_steps=25,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1_avg',
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=SEED,
    label_names=['labels_sentiment', 'labels_intent', 'labels_issue', 'labels_priority'],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('Trainer ready')


In [ ]:
# Train
train_result = trainer.train()
print(train_result)


In [ ]:
# Evaluate
val_metrics = trainer.evaluate(eval_dataset=val_ds)
print('Validation metrics:')
print(json.dumps(val_metrics, indent=2))

test_pred = trainer.predict(test_ds)
test_metrics = compute_metrics(test_pred)
print('Test metrics:')
print(json.dumps(test_metrics, indent=2))


In [ ]:
# Calibration (temperature scaling per task)

def softmax_np(logits: np.ndarray) -> np.ndarray:
    logits = logits - logits.max(axis=-1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=-1, keepdims=True)


def nll_with_temp(logits: np.ndarray, labels: np.ndarray, temp: float) -> float:
    probs = softmax_np(logits / temp)
    probs = np.clip(probs, 1e-9, 1.0)
    return float(-np.mean(np.log(probs[np.arange(len(labels)), labels])))


def tune_temperature(logits: np.ndarray, labels: np.ndarray) -> float:
    candidates = np.linspace(0.6, 3.0, 25)
    losses = [nll_with_temp(logits, labels, t) for t in candidates]
    return float(candidates[int(np.argmin(losses))])

val_pred = trainer.predict(val_ds)
val_logits = val_pred.predictions
val_labels = val_pred.label_ids

if not (isinstance(val_logits, (tuple, list)) and isinstance(val_labels, (tuple, list)) and len(val_logits) == 4 and len(val_labels) == 4):
    raise ValueError('Unexpected val prediction format for calibration.')

temperatures = {
    'sentiment': tune_temperature(val_logits[0], val_labels[0]),
    'intent': tune_temperature(val_logits[1], val_labels[1]),
    'issueType': tune_temperature(val_logits[2], val_labels[2]),
    'priority': tune_temperature(val_logits[3], val_labels[3]),
}

(OUTPUT_DIR / 'temperature_scaling.json').write_text(json.dumps(temperatures, indent=2), encoding='utf-8')
print('Calibrated temperatures:', temperatures)


In [ ]:
# Detailed reports, hard-case metrics, confusion matrices
preds = test_pred.predictions
labels = test_pred.label_ids
if not (isinstance(preds, (tuple, list)) and isinstance(labels, (tuple, list)) and len(preds) == 4 and len(labels) == 4):
    raise ValueError('Unexpected test prediction format.')

y_sent, y_intent, y_issue, y_prio = labels

p_sent = np.argmax(preds[0], axis=-1)
p_intent = np.argmax(preds[1], axis=-1)
p_issue = np.argmax(preds[2], axis=-1)
p_prio = np.argmax(preds[3], axis=-1)

reports = {
    'sentiment': classification_report(y_sent, p_sent, target_names=SENTIMENT_LABELS, output_dict=True),
    'intent': classification_report(y_intent, p_intent, target_names=INTENT_LABELS, output_dict=True),
    'issueType': classification_report(y_issue, p_issue, target_names=ISSUE_LABELS, output_dict=True),
    'priority': classification_report(y_prio, p_prio, target_names=PRIORITY_LABELS, output_dict=True),
}

confusions = {
    'sentiment': confusion_matrix(y_sent, p_sent).tolist(),
    'intent': confusion_matrix(y_intent, p_intent).tolist(),
    'issueType': confusion_matrix(y_issue, p_issue).tolist(),
    'priority': confusion_matrix(y_prio, p_prio).tolist(),
}

hard_idx = [i for i, m in enumerate(test_meta) if bool(m.get('hard_case', False))]
if len(hard_idx) > 0:
    hard_metrics = {
        'count': len(hard_idx),
        'sentiment_macro_f1': f1_score(np.array(y_sent)[hard_idx], np.array(p_sent)[hard_idx], average='macro'),
        'intent_macro_f1': f1_score(np.array(y_intent)[hard_idx], np.array(p_intent)[hard_idx], average='macro'),
        'issue_macro_f1': f1_score(np.array(y_issue)[hard_idx], np.array(p_issue)[hard_idx], average='macro'),
        'priority_macro_f1': f1_score(np.array(y_prio)[hard_idx], np.array(p_prio)[hard_idx], average='macro'),
    }
    hard_metrics['macro_f1_avg'] = float(np.mean([
        hard_metrics['sentiment_macro_f1'],
        hard_metrics['intent_macro_f1'],
        hard_metrics['issue_macro_f1'],
        hard_metrics['priority_macro_f1'],
    ]))
else:
    hard_metrics = {'count': 0}

(OUTPUT_DIR / 'classification_reports.json').write_text(json.dumps(reports, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'confusion_matrices.json').write_text(json.dumps(confusions, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'hard_case_metrics.json').write_text(json.dumps(hard_metrics, indent=2), encoding='utf-8')

print('Saved classification reports, confusion matrices, and hard-case metrics.')
print(json.dumps(hard_metrics, indent=2))


In [ ]:
# Save model and summary
MODEL_OUT = OUTPUT_DIR / 'model'
MODEL_OUT.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(MODEL_OUT)
tokenizer.save_pretrained(MODEL_OUT)

summary = {
    'run_id': RUN_ID,
    'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
    'model_name': MODEL_NAME,
    'seed': SEED,
    'max_length': MAX_LENGTH,
    'train_size': len(train_ds),
    'val_size': len(val_ds),
    'test_size': len(test_ds),
    'loss_weights': LOSS_WEIGHTS,
    'class_weights': {
        'sentiment': sentiment_class_weights.tolist(),
        'priority': priority_class_weights.tolist(),
    },
    'calibration_temperatures': json.loads((OUTPUT_DIR / 'temperature_scaling.json').read_text(encoding='utf-8')),
    'training_args': {
        'epochs': training_args.num_train_epochs,
        'train_batch_size': training_args.per_device_train_batch_size,
        'eval_batch_size': training_args.per_device_eval_batch_size,
        'learning_rate': training_args.learning_rate,
        'weight_decay': training_args.weight_decay,
    },
    'val_metrics': val_metrics,
    'test_metrics': test_metrics,
}

(OUTPUT_DIR / 'training_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')

zip_base = str(OUTPUT_DIR / f'insighta_distilbert_multitask_{RUN_ID}')
zip_path = shutil.make_archive(zip_base, 'zip', root_dir=OUTPUT_DIR)
print('Saved model to:', MODEL_OUT)
print('Wrote summary to:', OUTPUT_DIR / 'training_summary.json')
print('Zip artifact:', zip_path)


In [ ]:
# Inference demo
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = trainer.model.to(DEVICE)
model.eval()

def softmax_1d(x):
    x = x - np.max(x)
    ex = np.exp(x)
    return ex / ex.sum()

def predict_texts(texts: List[str]):
    batch = tokenizer(texts, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors='pt')
    batch = {k: v.to(DEVICE) for k, v in batch.items()}

    with torch.no_grad():
        out = model(**batch)

    logits_sent, logits_intent, logits_issue, logits_prio = [x.cpu().numpy() for x in out['logits']]
    temperatures = json.loads((OUTPUT_DIR / 'temperature_scaling.json').read_text(encoding='utf-8'))

    rows = []
    for i, text in enumerate(texts):
        ps = softmax_1d(logits_sent[i] / temperatures['sentiment'])
        pi = softmax_1d(logits_intent[i] / temperatures['intent'])
        pit = softmax_1d(logits_issue[i] / temperatures['issueType'])
        pp = softmax_1d(logits_prio[i] / temperatures['priority'])

        rows.append({
            'text': text,
            'sentiment': SENTIMENT_LABELS[int(np.argmax(ps))],
            'detectedIntent': INTENT_LABELS[int(np.argmax(pi))],
            'issueType': ISSUE_LABELS[int(np.argmax(pit))],
            'priority': PRIORITY_LABELS[int(np.argmax(pp))],
            'confidence': float(min(ps.max(), pi.max(), pit.max(), pp.max())),
        })
    return rows

sample_texts = [
    'I appreciate your support team, but this unresolved billing and payment mismatch is urgent for me.',
    'Claim rejected again and I need a proper review plus status update immediately.',
    'Portal crashes while changing policy details and now I cannot proceed.',
]

preds = predict_texts(sample_texts)
print(json.dumps(preds, indent=2))
